# Encode train-clean-360 on Colab

Encodes ~104K additional audio files into articulatory features.
- Features saved continuously to Google Drive (every 100 files)
- Survives session disconnects — resumes from Drive on restart
- LibriSpeech is re-downloaded each session (~10 min, unavoidable)

**Expected runtime:** ~15-25 hours on paid T4
**Compute units:** ~40-50 units

**Runtime setup:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 0: GPU check + Drive mount
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_FEATURES = '/content/drive/MyDrive/articulatory_tts/features_360'
os.makedirs(DRIVE_FEATURES, exist_ok=True)

existing = len([f for f in os.listdir(DRIVE_FEATURES) if f.endswith('.npz') and f != 'norm_stats.npz'])
print(f'Already encoded on Drive: {existing} files')

In [ ]:
# Step 1: Install SPARC
!pip install -q speech-articulatory-coding soundfile

In [ ]:
# Step 2: Download and extract LibriSpeech train-clean-360
import os, glob

if not os.path.exists('LibriSpeech/train-clean-360'):
    print('Downloading train-clean-360 (13GB)...')
    !wget -q --show-progress http://www.openslr.org/resources/12/train-clean-360.tar.gz
    print('Extracting...')
    !tar xzf train-clean-360.tar.gz
    !rm train-clean-360.tar.gz
else:
    print('Already downloaded.')

files = sorted(glob.glob('LibriSpeech/train-clean-360/**/*.flac', recursive=True))
print(f'train-clean-360: {len(files)} files')

In [ ]:
# Step 3: Load SPARC on GPU
from sparc import load_model
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading SPARC on {device}...')
coder = load_model('en', device=device)
print('SPARC loaded.')

# Quick warmup on one file (output discarded) to get stable timing
_ = coder.encode(files[0])
t0 = time.time()
for f in files[1:4]:
    _ = coder.encode(f)
avg = (time.time() - t0) / 3
print(f'Speed: {avg:.2f}s per file')
print(f'Estimated total: {avg * len(files) / 3600:.1f} hours for {len(files)} files')

In [ ]:
# Step 4: Encode — saves locally, syncs to Drive every 100 files
import numpy as np
import shutil
from pathlib import Path
from tqdm.notebook import tqdm

LOCAL_FEATURES = '/content/features'
os.makedirs(LOCAL_FEATURES, exist_ok=True)

# Check what's already on Drive (use listdir, faster than glob for large dirs)
drive_done = {
    f[:-4] for f in os.listdir(DRIVE_FEATURES)
    if f.endswith('.npz') and f != 'norm_stats.npz'
}
print(f'Already on Drive: {len(drive_done)}')

# Track which local files have been successfully synced to Drive
local_synced = set()

success = 0
skipped = 0
failed = 0
local_unsaved = 0
sync_failures = 0
t_start = time.time()


def sync_to_drive():
    """Copy new local files to Drive. Returns number of failures."""
    global local_synced, sync_failures
    failures = 0
    for lf in Path(LOCAL_FEATURES).glob('*.npz'):
        if lf.name in local_synced:
            continue
        drive_path = Path(DRIVE_FEATURES) / lf.name
        if drive_path.exists():
            local_synced.add(lf.name)
            continue
        try:
            shutil.copy2(str(lf), str(drive_path))
            local_synced.add(lf.name)
        except Exception as e:
            failures += 1
            sync_failures += 1
            if sync_failures < 5:
                print(f'  SYNC FAIL {lf.name}: {e}')
    return failures


for f in tqdm(files, desc='Encoding'):
    utt_id = Path(f).stem

    if utt_id in drive_done:
        skipped += 1
        continue

    local_path = Path(LOCAL_FEATURES) / f'{utt_id}.npz'
    if local_path.exists():
        skipped += 1
        local_unsaved += 1
    else:
        try:
            code = coder.encode(f)
            ema = np.asarray(code['ema'], dtype=np.float32)
            pitch = np.asarray(code['pitch'], dtype=np.float32).squeeze(-1)
            loudness = np.asarray(code['loudness'], dtype=np.float32).squeeze(-1)
            spk_emb = np.asarray(code['spk_emb'], dtype=np.float32)

            min_len = min(ema.shape[0], pitch.shape[0], loudness.shape[0])
            ema, pitch, loudness = ema[:min_len], pitch[:min_len], loudness[:min_len]

            np.savez_compressed(str(local_path), ema=ema, pitch=pitch, loudness=loudness, spk_emb=spk_emb)
            success += 1
            local_unsaved += 1
        except Exception as e:
            if failed < 5:
                print(f'FAILED {utt_id}: {e}')
            failed += 1

    # Sync to Drive every 100 new files
    if local_unsaved >= 100:
        sync_to_drive()
        local_unsaved = 0

    # Progress every 1000 files
    if (success + skipped) % 1000 == 0 and success > 0:
        elapsed = time.time() - t_start
        rate = success / max(elapsed, 1)
        remaining = (len(files) - success - skipped - failed) / max(rate, 0.01)
        total_done = len(drive_done) + success
        print(f'  [{total_done}/{len(files)}] {success} new, {skipped} skipped, ~{remaining/60:.0f}min left')

# Final sync
if local_unsaved > 0:
    print(f'\nFinal sync: {local_unsaved} files to Drive...')
    sync_to_drive()

# Retry any that failed to sync
max_retries = 3
for attempt in range(max_retries):
    unsynced = [
        lf.name for lf in Path(LOCAL_FEATURES).glob('*.npz')
        if lf.name not in local_synced
    ]
    if not unsynced:
        break
    print(f'Retry {attempt + 1}: syncing {len(unsynced)} leftover files...')
    sync_to_drive()

elapsed = time.time() - t_start
total_on_drive = len([f for f in os.listdir(DRIVE_FEATURES) if f.endswith('.npz') and f != 'norm_stats.npz'])
print(f'\nDone in {elapsed/3600:.1f}h: {success} new, {skipped} skipped, {failed} encode fails, {sync_failures} sync fails')
print(f'Total on Drive: {total_on_drive} / {len(files)}')
if total_on_drive < len(files) and success + skipped + failed == len(files):
    print(f'Note: {len(files) - total_on_drive} files missing on Drive — check sync errors above')

## After encoding completes

### 1. Download features from Google Drive

Go to drive.google.com → Navigate to `articulatory_tts/features_360/` → Right-click → Download (auto-zips)

### 2. Extract on your Mac

```bash
cd ~/projects/articulatory-tts/data
mkdir -p features_360_raw
cd features_360_raw
unzip ~/Downloads/features_360-*.zip

# If the zip extracted to a nested folder, flatten it:
cd ~/projects/articulatory-tts/data/features_360_raw
find . -mindepth 2 -name '*.npz' -exec mv {} . \;
find . -type d -empty -delete
```

### 3. Merge with existing features (avoids 'argument list too long')

```bash
cd ~/projects/articulatory-tts
mkdir -p data/features_combined_360

# Symlink existing 31K features
find data/features_all -maxdepth 1 -name '*.npz' -exec ln -sf "$(pwd)/{}" data/features_combined_360/ \;

# Symlink new 360 features
find data/features_360_raw -maxdepth 1 -name '*.npz' -exec ln -sf "$(pwd)/{}" data/features_combined_360/ \;

echo "Combined: $(ls data/features_combined_360/*.npz | wc -l) files"
```

### 4. Rebuild dataset + retrain

Then rebuild MFA dataset pointing at the combined features, retrain VQ + transformer.